In [6]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import torch.optim as optim
import torchvision.transforms as transforms
import torchvision
import numpy as np

from sklearn.model_selection import train_test_split

# Model Definition

In [2]:
class LeNet5(nn.Module):
    def __init__(self, num_classes=10, input_channels=1):
        super(LeNet5, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(input_channels, 6, kernel_size=5),
            nn.Tanh(),
            nn.AvgPool2d(kernel_size=2, stride=2),
            nn.Conv2d(6, 16, kernel_size=5),
            nn.Tanh(),
            nn.AvgPool2d(kernel_size=2, stride=2)
        )
        self.classifier = nn.Sequential(
            nn.Linear(16 * 5 * 5, 120),
            nn.Tanh(),
            nn.Linear(120, 84),
            nn.Tanh(),
            nn.Linear(84, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, 1) 
        x = self.classifier(x)
        return x

In [5]:
model = LeNet5(num_classes = 10, input_channels = 1)

# Dataset

In [8]:
transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])
train_dataset = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=1000, shuffle=False)

100%|██████████| 9.91M/9.91M [00:07<00:00, 1.25MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 101kB/s]
100%|██████████| 1.65M/1.65M [00:04<00:00, 398kB/s] 
100%|██████████| 4.54k/4.54k [00:00<00:00, 1.11MB/s]


# Train

In [9]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.9)

In [10]:
for epoch in range(1, 11):
    model.train()
    running_loss = 0.0
    for batch_idx, (data, target) in enumerate(train_loader, 1):
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        if batch_idx % 100 == 0:
            print(f'Epoch {epoch} [{batch_idx * len(data)}/{len(train_loader.dataset)}]  Loss: {running_loss/100:.4f}')
            running_loss = 0.0
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            outputs = model(data)
            _, predicted = torch.max(outputs, 1)
            total += target.size(0)
            correct += (predicted == target).sum().item()
    print(f'Epoch {epoch}  Test Accuracy: {100. * correct / total:.2f}%')


Epoch 1 [6400/60000]  Loss: 1.4850
Epoch 1 [12800/60000]  Loss: 0.4575
Epoch 1 [19200/60000]  Loss: 0.3423
Epoch 1 [25600/60000]  Loss: 0.3103
Epoch 1 [32000/60000]  Loss: 0.2543
Epoch 1 [38400/60000]  Loss: 0.2273
Epoch 1 [44800/60000]  Loss: 0.1846
Epoch 1 [51200/60000]  Loss: 0.1607
Epoch 1 [57600/60000]  Loss: 0.1542
Epoch 1  Test Accuracy: 96.41%
Epoch 2 [6400/60000]  Loss: 0.1253
Epoch 2 [12800/60000]  Loss: 0.1172
Epoch 2 [19200/60000]  Loss: 0.0988
Epoch 2 [25600/60000]  Loss: 0.0965
Epoch 2 [32000/60000]  Loss: 0.0833
Epoch 2 [38400/60000]  Loss: 0.0861
Epoch 2 [44800/60000]  Loss: 0.0858
Epoch 2 [51200/60000]  Loss: 0.0868
Epoch 2 [57600/60000]  Loss: 0.0881
Epoch 2  Test Accuracy: 98.15%
Epoch 3 [6400/60000]  Loss: 0.0731
Epoch 3 [12800/60000]  Loss: 0.0683
Epoch 3 [19200/60000]  Loss: 0.0604
Epoch 3 [25600/60000]  Loss: 0.0659
Epoch 3 [32000/60000]  Loss: 0.0599
Epoch 3 [38400/60000]  Loss: 0.0652
Epoch 3 [44800/60000]  Loss: 0.0556
Epoch 3 [51200/60000]  Loss: 0.0547
Epoch

# Test

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
num_example = 10
test_sample = x_test[:10, :]
predict_test_sample = model.forward(test_sample)

# visualize 

imgs = []
for img in test_sample:
    img = img.reshape(32, 32)
    imgs.append(img)

titles = []
for title in predict_test_sample:
    titles.append(np.argmax(title))

# Create a figure with 2 rows and 5 columns
fig, axes = plt.subplots(2, 5, figsize=(15, 6))

# axes is a 2×5 array — flatten it so we can zip with images & titles
for ax, img, title in zip(axes.flat, imgs, titles):
    ax.imshow(img, cmap='gray')    # or remove cmap= if your images are RGB
    ax.set_title(f"predict : {title}")
    ax.axis('off')                 # turn off axis ticks/labels

plt.tight_layout()
plt.show()